Implementazione del gradiente semplice

In [ ]:
import torch

# Si assume che 'model' sia una rete convoluzionale pre-addestrata
# in modalita' di valutazione, e che 'input_tensor' abbia forma
# (1, 3, H, W) e sia normalizzato secondo le statistiche del
# dataset di addestramento.
model.eval()
input_tensor.requires_grad_(True)

output = model(input_tensor)          # forward pass: logits (1, C)
target_class = output.argmax(dim=1)   # classe predetta

# Backward pass rispetto al logit della classe target
score = output[0, target_class]
model.zero_grad()
score.backward()

# Mappa di salienza: massimo valore assoluto sui canali RGB
saliency, _ = torch.max(input_tensor.grad.data.abs(), dim=1)
saliency = saliency.squeeze().cpu().numpy()  # forma (H, W)

Implementazione di Grad-CAM:

In [ ]:
import torch
import torch.nn.functional as F

activations = {}
gradients = {}

def forward_hook(module, input, output):
    activations['value'] = output.detach()

def backward_hook(module, grad_input, grad_output):
    gradients['value'] = grad_output[0].detach()

# Si registra l'hook sull'ultimo strato convoluzionale del backbone
target_layer = model.layer4[-1]
target_layer.register_forward_hook(forward_hook)
target_layer.register_full_backward_hook(backward_hook)

model.eval()
output = model(input_tensor)          # forward: popola 'activations'
target_class = output.argmax(dim=1)

model.zero_grad()
output[0, target_class].backward()    # backward: popola 'gradients'

A = activations['value'][0]           # forma (K, u, v)
grad = gradients['value'][0]          # forma (K, u, v)
alpha = grad.mean(dim=(1, 2))         # GAP del gradiente: alpha_k^c

gradcam_map = F.relu((alpha[:, None, None] * A).sum(dim=0))

# Upsampling bilineare alla risoluzione dell'immagine originale
gradcam_map = gradcam_map.unsqueeze(0).unsqueeze(0)
gradcam_map = F.interpolate(
    gradcam_map,
    size=input_tensor.shape[-2:],
    mode='bilinear',
    align_corners=False
)
gradcam_map = gradcam_map.squeeze().cpu().numpy()

Implementazione RISE

In [ ]:
import torch
import torch.nn.functional as F

def generate_rise_masks(n_masks, low_res, img_size, p=0.5):
    grid = (torch.rand(n_masks, 1, low_res, low_res) < p).float()
    masks = F.interpolate(
        grid, size=img_size, mode='bilinear', align_corners=False
    )
    return masks  # forma (n_masks, 1, H, W), valori in [0, 1]

model.eval()
masks = generate_rise_masks(
    n_masks=2000, low_res=7, img_size=input_tensor.shape[-2:]
)
saliency_map = torch.zeros(input_tensor.shape[-2:])

with torch.no_grad():
    for i in range(masks.shape[0]):
        masked_input = input_tensor * masks[i]
        score = F.softmax(model(masked_input), dim=1)[0, target_class]
        saliency_map += score.item() * masks[i, 0]

saliency_map /= (masks.shape[0] * masks.mean())